### Import dependencies

In [ ]:
# Model parameters used throughout the notebooks
k_on_0 = 3.47e-4
k_off_0 = 2.68e-4 
f_1e = 2.49e-1

# import utility functions
include("src/utils.jl");

# import functions that help calculate stability equilibria
include("src/f_vectors_k_vectors_nvectors.jl");
# This function return variables that act as look-up tables for stability that can be used by functions of f_vectors_k_vectors_nvectors.jl
K, F, n_stable_store, n_unstable_store = setup_stores(k_off_0 / k_on_0);

# import functions that define loading patterns
include("src/loading_patterns.jl");

# import functions that define the analytical model
include("src/analytical_model.jl");

# import functions that define the stochastic model
include("src/stochastic_model.jl");

### Analytical model: Calculate recovery timescale, fracture timescale and simulate maps

In [ ]:
gamma_high = 0.15
gamma_low = 0.015
duration = 20000

# get time to recovery
time_to_recovery = get_recovery_time(gamma_low, k_on_0, k_off_0, f_1e, duration, false)

# get time to fracture
time_to_fracture = get_fracture_time(gamma_high, k_on_0, k_off_0, f_1e, duration, gamma_low)

# get stable and unstable n at low gamma to initialize cyclic simulations
n_stable, n_unstable = critical_n(k_off_0/k_on_0, gamma_low/f_1e, K, F, n_stable_store, n_unstable_store, true)
n_init = n_stable


# proportional ranges of high and low time periods relative to their respective timescales
time_high_ratio = range(0.01, 1.5, length=40)
time_low_ratio = range(0.01, 4.5, length=40)

# run simulations over grid of time ratios
cyclic_time_to_fracture = zeros(length(time_high_ratio), length(time_low_ratio))
@suppress for t1 in 1:length(time_high_ratio)
    @threads for t2 in 1:length(time_low_ratio)
        # caluclate duration to spend 20x time_to_fracture at high tension for current t_high and t_low
        t_high = time_high_ratio[t1]*time_to_fracture
        t_low = time_low_ratio[t2]*time_to_recovery
        objective_time_high = 20 * time_to_fracture
        period = t_high + t_low
        n_periods = ceil(objective_time_high / t_high)
        duration = n_periods * period

        loading = square_cycle_uneven_new(gamma_high, gamma_low, time_high_ratio[t1]*time_to_fracture, time_low_ratio[t2]*time_to_recovery)
        sol = solve_dn_dt(n_init, k_on_0, k_off_0, f_1e, loading, duration, 1.0)
        fracture_time = find_fracture(sol)
        cyclic_time_to_fracture[t1, t2] = fracture_to_t_high(fracture_time, t_high, t_low) # find how long spent at high tension before fracture                            
    end
end

In [ ]:
# rescale data from 0 to 1
cyclic_time_to_fracture_plot = adjoint(cyclic_time_to_fracture)
cyclic_time_to_fracture_plot = (cyclic_time_to_fracture_plot .- time_to_fracture) ./ cyclic_time_to_fracture_plot

# Plot heatmap
x = vec(time_high_ratio) .* time_to_fracture
y = vec(time_low_ratio) .* time_to_recovery
plot(size=(450, 400), guidefontsize=16, tickfontsize=14, legendfontsize=14, margin=3Plots.mm)
heatmap!(x./1000,y./1000, cyclic_time_to_fracture_plot, color=:viridis, xlabel="\$t_{high} (10^3 s)\$", ylabel="\$t_{low} (10^3 s)\$", cbar=true, clims=(0, 1))
xlims!(0, maximum(x./1000))
ylims!(0, maximum(y./1000))

![](figures/cyclic_time_to_fracture_heatmap.png "Title")


### Stochastic model: Calculate recovery timescale, fracture timescale and simulate maps

In [ ]:
gamma_high = 0.15
gamma_low = 0.015
duration = 20000

num_sim = 100
n = 100
l = 1

time_to_recovery_stochastic = get_recovery_time_stochastic(gamma_low, k_on_0, k_off_0, f_1e, duration, false, false, 1.0, num_sim, n)

time_to_fracture_stochastic = get_fracture_time_stochastic(gamma_high, k_on_0, k_off_0, f_1e, duration, gamma_low, num_sim, n, true)

# get stable and unstable n at low gamma to initialize cyclic simulations
n_stable, n_unstable = critical_n(k_off_0/k_on_0, gamma_low/f_1e, K, F, n_stable_store, n_unstable_store, true)
n_init = n_stable

# proportional ranges of high and low time periods relative to their respective timescales
time_high_ratio = range(0.01, 1.5, length=40)
time_low_ratio = range(0.01, 4.5, length=40)

# run simulations over grid of time ratios
cyclic_time_to_fracture_stochastic = zeros(length(time_high_ratio), length(time_low_ratio), num_sim)

for t1 in 1:length(time_high_ratio)
    for t2 in 1:length(time_low_ratio)
        # get duration to spend 20x time_to_fracture at high tension for current t_high and t_low
        objective_time_high = 20 * time_to_fracture_stochastic
        t_high = time_high_ratio[t1]*time_to_fracture_stochastic
        t_low = time_low_ratio[t2]*time_to_recovery_stochastic
        period = t_high + t_low
        n_periods = ceil(objective_time_high / t_high)
        duration = n_periods * period

        # set dt to 1/100 of the minimum of time_high and time_low periods for fast simulations
        dt = minimum([t_high, t_low])/100

        # modify dt so that a multiple of dt is equal to the duration
        dt = duration/round(Int, duration/dt)
        
        # define cyclic loading for stochastic model
        loading = square_cycle_uneven_new(gamma_high, gamma_low, t_high, t_low)
        tension_bond, n_timesteps = loading_stochastic_model(loading, n, duration, dt)
        
        @threads for sim = 1:1:num_sim   # number of simulations
            model = SlipBondModel((k_on_0=k_on_0,), (k_off_0=k_off_0, f_1e=f_1e)) 
            x = Cluster(n, l, model, :force_global)
            x = initialise_bonds_state(x, n_init)
            _, _, fracture_time, _ = runcluster(x, tension_bond, dt, max_steps = n_timesteps, verbose=false)
            cyclic_time_to_fracture_stochastic[t1, t2, sim] = fracture_to_t_high(fracture_time, t_high, t_low)
        end
    end
end

In [ ]:
# rescale data from 0 to 1
cyclic_time_to_fracture_plot = adjoint(mean(cyclic_time_to_fracture_stochastic, dims=3)[:,:,1])
cyclic_time_to_fracture_plot = (cyclic_time_to_fracture_plot .- time_to_fracture_stochastic) ./ cyclic_time_to_fracture_plot

# Plot heatmap
x = vec(time_high_ratio) .* time_to_fracture_stochastic
y = vec(time_low_ratio) .* time_to_recovery_stochastic
plot(size=(450, 400), guidefontsize=16, tickfontsize=14, legendfontsize=14, margin=3Plots.mm)
heatmap!(x./1000,y./1000, cyclic_time_to_fracture_plot, color=:viridis, xlabel="\$t_{high} (10^3 s)\$", ylabel="\$t_{low} (10^3 s)\$", cbar=true, clims=(0, 1))
xlims!(0, maximum(x./1000))
ylims!(0, maximum(y./1000))